In [1]:
import sys, glob
sys.path.append("../src")
import pandas as pd, numpy as np, folium
from clean import haversine_km, remove_jumps

f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = pd.read_csv(f).drop_duplicates(["vehicle_id", "timestamp"])
stops = pd.read_csv("../data/raw/delhi_bus_gtfs/stops.txt")

clean = pd.concat([remove_jumps(g) for _, g in pos.groupby("vehicle_id")])
print(len(pos), "->", len(clean), "points after cleaning,", clean.vehicle_id.nunique(), "buses")

117335 -> 117048 points after cleaning, 5955 buses


In [2]:
clean = clean.sort_values(["vehicle_id", "timestamp"])
nxt = clean.groupby("vehicle_id")[["lat", "lon", "timestamp"]].shift(-1)
clean["step_km"] = haversine_km(clean.lat, clean.lon, nxt.lat, nxt.lon)
clean["gap_s"] = nxt.timestamp - clean.timestamp

dwell = clean[(clean.step_km < 0.03) & (clean.gap_s <= 120)]
print(len(dwell), "stationary points")

34769 stationary points


In [3]:
from sklearn.cluster import DBSCAN
coords = np.radians(dwell[["lat", "lon"]].values)
db = DBSCAN(eps=0.03/6371, min_samples=3, metric="haversine", algorithm="ball_tree").fit(coords)
dwell = dwell.assign(cluster=db.labels_)

detected = (dwell[dwell.cluster >= 0].groupby("cluster")
            .agg(lat=("lat", "mean"), lon=("lon", "mean"),
                 n_points=("lat", "size"), n_buses=("vehicle_id", "nunique")))
detected = detected[detected.n_buses >= 2]
print(len(detected), "detected stops")

788 detected stops


In [4]:
from sklearn.neighbors import BallTree
tree = BallTree(np.radians(stops[["stop_lat", "stop_lon"]].values), metric="haversine")
dist, _ = tree.query(np.radians(detected[["lat", "lon"]].values), k=1)
detected["nearest_stop_m"] = dist[:, 0] * 6371000

print(f"{(detected.nearest_stop_m <= 50).mean():.0%} of detected stops are within 50 m of a real bus stop")
detected.nearest_stop_m.describe()

44% of detected stops are within 50 m of a real bus stop


count      788.000000
mean       200.161159
std       1687.204883
min          1.574653
25%         25.507334
50%         60.824013
75%        125.488355
max      36974.905910
Name: nearest_stop_m, dtype: float64

In [6]:
m = folium.Map(location=[28.61, 77.21], zoom_start=12, tiles="OpenStreetMap")
for _, s in detected.iterrows():
    color = "green" if s.nearest_stop_m <= 50 else "orange"
    folium.CircleMarker([s.lat, s.lon], radius=4, color=color, fill=True,
                        tooltip=f"{s.n_buses} buses, {s.nearest_stop_m:.0f} m to nearest stop").add_to(m)
m

In [7]:
for min_buses in [2, 3, 5]:
    d = detected[(detected.n_buses >= min_buses) & (detected.nearest_stop_m <= 500)]
    print(f"min_buses={min_buses}: {len(d)} stops | "
          f"within 50 m: {(d.nearest_stop_m <= 50).mean():.0%} | "
          f"within 100 m: {(d.nearest_stop_m <= 100).mean():.0%}")

min_buses=2: 769 stops | within 50 m: 45% | within 100 m: 66%
min_buses=3: 567 stops | within 50 m: 49% | within 100 m: 71%
min_buses=5: 298 stops | within 50 m: 47% | within 100 m: 74%


In [ ]:
import osmnx as ox
signals = ox.features_from_point((28.61, 77.21), tags={"highway": "traffic_signals"}, dist=30000)
sig = signals[signals.geometry.geom_type == "Point"]
print(len(sig), "traffic signals mapped")

In [ ]:
sig_tree = BallTree(np.radians(np.c_[sig.geometry.y, sig.geometry.x]), metric="haversine")
d, _ = sig_tree.query(np.radians(detected[["lat", "lon"]].values), k=1)
detected["nearest_signal_m"] = d[:, 0] * 6371000